<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/02_openai_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 2: Talking to an LLM through the OpenAI API

In notebook 1 we ran GPT-2 ourselves, on Colab's GPU. The biggest models (OpenAI's GPT, Anthropic's Claude, Google's Gemini) do not fit there, and their weights are not public. You use them through an **API**: your program sends a text over the internet, the provider runs the model on its own GPUs and sends the answer back. You pay for every **token**, in and out.

**What you will do:**
- send your first request, and see what it costs
- use a system prompt and hold a conversation
- use the LLM as a data science tool: rate Czech movie reviews and get the answers back as structured data

**Contents:** 0. API key · 1. The first request · 2. Tokens, price and reasoning · 3. System prompt and conversation · 4. Structured output: rating movie reviews · 5. Other providers · 6. Summary

No GPU needed: the model runs at OpenAI.

---
## 0. API key

The API needs a **key**: a secret that says who pays. Create one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys). The API is paid separately from a ChatGPT subscription: you top up credit in advance (at least $5). Everything in this notebook costs about one cent.

Store the key the same way as `HF_TOKEN` in notebook 1: 🔑 in the left sidebar → *Add new secret* → Name `OPENAI_API_KEY` → switch on *Notebook access*.

⚠️ Never paste the key into a cell. Whoever finds it can spend your money. On the OpenAI platform you can also set a monthly budget limit.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-5.6-luna"        # a small and cheap OpenAI model

---
## 1. The first request

A request is a list of **messages**. Each message has a **role** (`user` is you) and a **content**:

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain in two sentences what a transformer is."}],
    reasoning_effort="low",
)
print(response.choices[0].message.content)

`reasoning_effort`: today's models **think** before they answer, i.e. they first generate hidden tokens with their reasoning. More thinking helps with hard problems, but it is slower and you pay for the hidden tokens too. The levels go from `"none"` over `"low"` and `"medium"` (the default) to `"high"` and more. We use `"low"`.

**Look inside.** The answer is an object with much more than the text:

In [ ]:
print(response.model_dump_json(indent=2))

- `choices`: the answer (you can ask for several at once)
- `finish_reason`: `"stop"` means the model ended by itself, `"length"` means it hit a token limit
- `usage`: how many tokens went in and out. This is what you pay for.

---
## 2. Tokens, price and reasoning

Prices are per million tokens. For `gpt-5.6-luna`: $0.20 for input tokens and $1.20 for output tokens ([pricing](https://developers.openai.com/api/docs/pricing)). Hidden reasoning tokens count as output tokens.

In [ ]:
PRICE_IN, PRICE_OUT = 0.20, 1.20          # USD per 1 million tokens

def cost(response):
    u = response.usage
    usd = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6
    print(f"{u.prompt_tokens} tokens in, {u.completion_tokens} tokens out "
          f"(of which {u.completion_tokens_details.reasoning_tokens} hidden reasoning): ${usd:.6f}")

cost(response)

Now a question that needs some thinking, with little and with a lot of reasoning:

In [ ]:
question = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with just the number."

for effort in ["low", "high"]:
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}], reasoning_effort=effort)
    print(f"reasoning_effort={effort!r}: {r.choices[0].message.content}")
    cost(r)

The more reasoning, the more hidden tokens the model writes (and you pay for) before the answer. You do not see them, only their number.

*(With `reasoning_effort="none"`, the API also lets you set the `temperature` and look at the next-token probabilities (`logprobs`), as we did with GPT-2 last week. With reasoning switched on, these settings are not allowed.)*

---
## 3. System prompt and conversation

A **system** message sets the rules for the whole conversation: a role, a style, a language, what to do and what not to do. Users of a chatbot do not see it.

In [ ]:
messages = [
    {"role": "system", "content": "You are a grumpy Czech grandmother. Always answer in Czech, in at most three sentences."},
    {"role": "user", "content": "What should I have for dinner?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, reasoning_effort="low")
answer = response.choices[0].message.content
print(answer)

**The model has no memory.** Every request starts from zero. A chatbot "remembers" the conversation because it sends the **whole history** every time. So we add the answer and our next message to the list:

In [ ]:
messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "And what about pizza?"})

response = client.chat.completions.create(model=MODEL, messages=messages, reasoning_effort="low")
print(response.choices[0].message.content)
cost(response)

The input tokens grew: we sent the first question and answer again. In a long conversation, every new message pays for the whole history.

**Try it:** Change the system prompt (a pirate, a strict teacher, an assistant that answers only in JSON...). Can you, as the user, convince the model to break the rules from the system prompt?

---
## 4. Structured output: rating movie reviews

LLMs are a handy data science tool: they can read thousands of texts and turn them into a table. Let's take 20 Czech user reviews from [ČSFD](https://www.csfd.cz), the Czech movie database (we load them with the `datasets` library, more about it in notebook 3), and ask the model to guess the number of stars, **without any training**.

To get the answer as data and not as free text, we describe the format with a `pydantic` class. The API then guarantees that the answer has exactly these fields:

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewRating(BaseModel):
    stars: int = Field(description="the rating the author gave, from 0 to 5 stars")
    sentiment: Literal["positive", "negative", "mixed"]
    summary: str = Field(description="one short sentence in English")

def rate(review):
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You read Czech user reviews of movies from ČSFD and guess how many stars (0-5) the author gave."},
            {"role": "user", "content": review},
        ],
        response_format=ReviewRating,
        reasoning_effort="low",
    )
    return response.choices[0].message.parsed

result = rate("Ze začátku jsem se nudil, ale konec to zachránil. Herci průměrní.")
print(result)
print(result.stars, result.sentiment)

`result` is a Python object, not text: `result.stars` is an `int`. Now 20 real reviews from the test split:

In [ ]:
from datasets import load_dataset

test = load_dataset("fewshot-goes-multilingual/cs_csfd-movie-reviews", split="test").shuffle(seed=0).select(range(20))

rows = []
for row in test:
    r = rate(row["comment"])
    rows.append({"title": row["item_title"], "true": row["rating_int"], "predicted": r.stars,
                 "sentiment": r.sentiment, "summary": r.summary})

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print("exactly right:            ", (df["true"] == df["predicted"]).mean())
print("off by at most one star:  ", ((df["true"] - df["predicted"]).abs() <= 1).mean())
df

We did not train anything: we only wrote one sentence of instructions (*zero-shot*). In notebook 4 we go the other way: we **train** a small BERT model on 2,000 labelled reviews. It needs labelled data, but then it runs on our own GPU and costs nothing per review.

**Try it:**
- Improve the system prompt (e.g. "ČSFD users are strict, 3 stars is an average movie") and measure again.
- Add a few example reviews with their true ratings into the messages (*few-shot*). Does it help?
- Add more fields to `ReviewRating`, e.g. `mentions_actors: bool`.

---
## 5. Other providers

Many providers copied OpenAI's API format, so the same `openai` library works with them. You only change the address (`base_url`), the key and the model name:

```python
client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=userdata.get("HF_TOKEN"))
client.chat.completions.create(model="openai/gpt-oss-120b", messages=...)
```

| provider | `base_url` | note |
|---|---|---|
| OpenAI | (default) | GPT models |
| Hugging Face Inference Providers | `https://router.huggingface.co/v1` | open models (gpt-oss, Llama, Qwen, ...), needs a token with the *Inference Providers* permission and paid credits |
| Ollama | `http://localhost:11434/v1` | open models running on your own computer, free |

Anthropic (Claude) and Google (Gemini) have their own libraries, and also offer an OpenAI-compatible endpoint.

*(OpenAI also has a newer **Responses API**, `client.responses.create(...)`, which it recommends for new projects. We use Chat Completions because it is the format that the other providers copy.)*

---
## 6. Summary

| | code |
|---|---|
| client | `client = OpenAI(api_key=...)` |
| a request | `client.chat.completions.create(model=..., messages=[...])` |
| the answer | `response.choices[0].message.content` |
| roles | `system` (rules), `user` (you), `assistant` (the model's earlier answers) |
| price | `response.usage`: tokens in and out |
| thinking | `reasoning_effort="none"`, `"low"`, ... `"high"` |
| structured output | `client.chat.completions.parse(..., response_format=PydanticClass)` |

**New words:** API, API key, prompt, system prompt, reasoning tokens, structured output, zero-shot, few-shot.

### Exercises
1. Rate 100 reviews instead of 20. How much did it cost? How much would all 30,000 reviews cost?
2. Compare `gpt-5.6-luna` with a bigger model, or with a higher `reasoning_effort` (see the [models page](https://developers.openai.com/api/docs/models)): is it more accurate? How much more expensive?
3. Ask the model for the same thing in Czech and in English, and compare the number of input tokens. (Remember lesson 04: which language is more expensive?)
4. With `reasoning_effort="none"`, add `logprobs=True, top_logprobs=10, max_completion_tokens=1` to a request and look at `response.choices[0].logprobs`: the next-token probabilities, as with GPT-2 last week.

### Further reading
- [OpenAI API documentation](https://developers.openai.com/api/docs), [structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers)